# CS781 Phase 4 — Experiment 5 — v4 BETTER
## Model3 Encoder–Decoder + Post-Residual Houlsby Adapters + Hierarchical CJPE

This version keeps the useful v3 fixes but changes the parts most likely responsible for the one-class collapse observed in the previous run.

- **True post-residual Houlsby adapters**: adapters act on the full hidden state after each Transformer residual update.
- Adapter-internal LayerNorm improves conditioning while zero-initialized up-projections preserve the exact pretrained function at step 0.
- Stronger chunk representation: encoder **last + mean + masked max** plus a **learnably gated** decoder cross-attention summary.
- The decoder contribution starts small instead of dominating the classifier.
- Milder class weighting, no label smoothing, and lower adapter/head LR to reduce collapse.
- DEV diagnostics print probability spread and positive rates so collapse is visible early.

All original Phase-1 Model3 parameters remain frozen. Only adapters, the decoder gate, and the CJPE hierarchical head are trainable.


## Why this should be more stable than v3

The previous run reached Macro-F1 ≈ 0.333 repeatedly, which indicates near one-class predictions. The training itself was numerically stable, so this version changes representation/adaptation rather than merely running longer.

The most important correction is the adapter placement. A standard Houlsby-style adapter operates on the hidden state **after** a Transformer sub-layer residual update. This gives the adapter access to the full contextual representation while still preserving the pretrained model exactly at initialization because the adapter up-projection is zero-initialized.

The decoder is still part of the experiment, but its single-query summary is gated with a learned sigmoid gate initialized near 0.12. This lets the model use the decoder if helpful without forcing a weak one-token decoder representation to dominate the encoder features.


In [1]:
# CELL 1 — imports/config
import os,re,gc,json,math,time,random,zipfile,subprocess,sys,hashlib,tempfile
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from sklearn.metrics import f1_score,precision_score,recall_score
from huggingface_hub import hf_hub_download,login

SEED=1234
NOTEBOOK_VERSION="v4_BETTER"

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()

def save_json(obj,path,**kw):
    path=Path(path); tmp=Path(str(path)+".tmp")
    with open(tmp,"w",encoding="utf-8") as f: json.dump(obj,f,**kw)
    os.replace(tmp,path)

def read_json(path):
    with open(path,encoding="utf-8") as f: return json.load(f)

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(1<<20),b""): h.update(b)
    return h.hexdigest()

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",DEVICE)
if DEVICE.type=="cuda":
    print("GPU:",torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32=True
    torch.backends.cudnn.allow_tf32=True
torch.set_float32_matmul_precision("high")

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN=None
assert HF_TOKEN,"HF_TOKEN missing in Kaggle Secrets"
login(token=HF_TOKEN)

HF_REPO="Exploration-Lab/CS-781-Capstone"
ROOT=Path("/kaggle/working/phase4_exp5_model3_adapters")
ROOT.mkdir(parents=True,exist_ok=True)

MODEL3_CKPT_OVERRIDE=None
TOKENIZER_OVERRIDE=None
EXPECTED_TOKENIZER_SHA256=None

# Exact expected Phase-1 Model3 architecture.
VOCAB_SIZE=24576
D_MODEL=512
N_ENCODER_LAYERS=9
N_DECODER_LAYERS=9
N_HEADS=8
FFN_DIM=2048
DROPOUT=0.0
MAX_ENCODER_POSITIONS=2048
MAX_DECODER_POSITIONS=1536

# Long-document representation.
N_CHUNKS=4
CHUNK_LEN=384
CHUNK_NAMES=["head","middle","evidence","tail"]
CASE_MODE="auto"
COLLAPSE_WS=True
EVIDENCE_CUES=[
    "appeal is allowed","appeal allowed",
    "appeal is dismissed","appeal dismissed",
    "petition is allowed","petition allowed",
    "petition is dismissed","petition dismissed",
    "we allow","we dismiss",
    "is hereby allowed","is hereby dismissed",
    "set aside","acquitted","convicted",
    "in the result","accordingly",
]
WEAK_CUES={"in the result","accordingly","set aside","convicted","acquitted"}

# Houlsby-style adapters.
ADAPTER_BOTTLENECK=128
ADAPTER_DROPOUT=0.05
ADAPTER_LR=3.0e-4

CHUNK_PROJ_DIM=384
HEAD_DROPOUT=0.15
HEAD_LR=3.0e-4

NUM_EPOCHS=3
EVAL_PER_EPOCH=4
PATIENCE_EVALS=6
MICRO_BATCH_SIZE=2
GRAD_ACCUM_STEPS=8
EVAL_BATCH_SIZE=6
WARMUP_RATIO=0.06
WEIGHT_DECAY=0.01
GRAD_CLIP=1.0
LABEL_SMOOTHING=0.0
CLASS_WEIGHT_POWER=0.50  # milder than full inverse-frequency weighting
USE_AMP=True
AMP_INIT_SCALE=4096.0
NUM_WORKERS=2

THRESHOLDS=np.round(np.arange(0.20,0.801,0.005),3)
THRESH_SMOOTH=7
RUN_REFIT=True
REFIT_MATCH_ORIGINAL_SCHEDULE=True

print("chunks:",N_CHUNKS,"x",CHUNK_LEN)
print("effective document batch:",MICRO_BATCH_SIZE*GRAD_ACCUM_STEPS)
print("adapter bottleneck:",ADAPTER_BOTTLENECK,"| adapter LR:",ADAPTER_LR)


Device: cuda
GPU: Tesla T4
chunks: 4 x 384
effective document batch: 16
adapter bottleneck: 128 | adapter LR: 0.0003


In [2]:
# CELL 2 — locate exact Model3 checkpoint/tokenizer robustly
def all_files():
    out=[]
    exclude=ROOT.resolve()
    for base in ["/kaggle/input","/kaggle/working"]:
        if not os.path.isdir(base): continue
        for root,_,names in os.walk(base):
            rr=Path(root).resolve()
            try:
                inside=rr.is_relative_to(exclude)
            except AttributeError:
                try: rr.relative_to(exclude); inside=True
                except ValueError: inside=False
            if inside: continue
            out.extend(os.path.join(root,n) for n in names)
    return out

files=all_files()

if MODEL3_CKPT_OVERRIDE:
    PHASE1_CKPT=str(MODEL3_CKPT_OVERRIDE)
else:
    ckpts=[
        p for p in files if p.lower().endswith(".pt")
        and (
            "encdec_full_best" in os.path.basename(p).lower()
            or ("model3" in p.lower() and "best" in os.path.basename(p).lower())
            or ("encdec" in p.lower() and "best" in os.path.basename(p).lower())
        )
    ]
    assert ckpts,"Attach Model3 encdec_full_best*.pt or set MODEL3_CKPT_OVERRIDE."
    PHASE1_CKPT=sorted(
        ckpts,
        key=lambda p:(sum(w in p.lower() for w in ["model3","encdec","encoder-decoder","phase1"]),p),
        reverse=True
    )[0]
    print("checkpoint candidates:")
    for p in ckpts: print("  -",p)

if TOKENIZER_OVERRIDE:
    TOKENIZER_FILE=str(TOKENIZER_OVERRIDE)
else:
    toks=[
        p for p in files
        if os.path.basename(p).lower().startswith("tokenizer")
        and p.lower().endswith(".json")
    ]
    assert toks,"Attach the Model3 tokenizer JSON or set TOKENIZER_OVERRIDE."
    ckpt_dir=str(Path(PHASE1_CKPT).parent)
    def tok_score(p):
        same=int(str(Path(p).parent)==ckpt_dir)
        common=len(os.path.commonpath([os.path.abspath(PHASE1_CKPT),os.path.abspath(p)]))
        hints=sum(w in p.lower() for w in ["model3","encdec","encoder","phase1"])
        return (same,common,hints,p)
    TOKENIZER_FILE=sorted(toks,key=tok_score,reverse=True)[0]
    print("tokenizer candidates:")
    for p in toks: print("  -",p)

assert os.path.isfile(PHASE1_CKPT)
assert os.path.isfile(TOKENIZER_FILE)

print("chosen checkpoint:",PHASE1_CKPT)
print("chosen tokenizer :",TOKENIZER_FILE)
print("checkpoint SHA256:",sha256_file(PHASE1_CKPT))
print("tokenizer SHA256 :",sha256_file(TOKENIZER_FILE))

if not TOKENIZER_OVERRIDE and len(toks)>1:
    hashes={p:sha256_file(p) for p in toks}
    for p,h in hashes.items(): print("  tokenizer",h[:12],p)
    if len(set(hashes.values()))>1:
        print("⚠️ Multiple DIFFERENT tokenizers attached. "
              "If the Phase-1 sanity check fails, set TOKENIZER_OVERRIDE explicitly.")

if EXPECTED_TOKENIZER_SHA256:
    assert sha256_file(TOKENIZER_FILE)==EXPECTED_TOKENIZER_SHA256,"Tokenizer SHA mismatch"


checkpoint candidates:
  - /kaggle/input/datasets/aaryant23/phase2-model3-ckpts/best_adapters_cjpe.pt
  - /kaggle/input/datasets/aaryant23/phase2-model3-ckpts/encdec_full_best (2).pt
tokenizer candidates:
  - /kaggle/input/datasets/aaryant23/phase2-model3-ckpts/tokenizer (1).json
chosen checkpoint: /kaggle/input/datasets/aaryant23/phase2-model3-ckpts/encdec_full_best (2).pt
chosen tokenizer : /kaggle/input/datasets/aaryant23/phase2-model3-ckpts/tokenizer (1).json
checkpoint SHA256: 28754f7ae29ab3e2c5789b979f7f3947c64d20052bf083a8899432f05a40d694
tokenizer SHA256 : 9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d


In [3]:
# CELL 3 — exact LegalBPE loader
BPE_SPECIALS=["<pad>","<unk>","<s>","</s>"]
class LegalBPE:
    def __init__(self,vocab_size=5000):
        self.vocab_size=int(vocab_size); self.merges=[]
        self.itos=list(BPE_SPECIALS); self.stoi={t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes={}
        for b in range(256):
            s=f"{b:02x}"; self.symbol_bytes[s]=bytes([b]); self.stoi[s]=len(self.itos); self.itos.append(s)
        self._reset_encode_cache()
    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)
    @staticmethod
    def word_to_symbols(word): return [f"{b:02x}" for b in word.encode("utf-8")]
    def _reset_encode_cache(self):
        self._ranks={(l,r):i for i,(l,r) in enumerate(self.merges)}
        self._word_cache={}; self._cache_nmerges=len(self.merges)
    def encode_word(self,word):
        if self._cache_nmerges!=len(self.merges): self._reset_encode_cache()
        hit=self._word_cache.get(word)
        if hit is not None: return list(hit)
        symbols=self.word_to_symbols(word)
        while len(symbols)>1:
            best=None; rank=None
            for j in range(len(symbols)-1):
                rr=self._ranks.get((symbols[j],symbols[j+1]))
                if rr is not None and (rank is None or rr<rank):
                    best=(symbols[j],symbols[j+1]); rank=rr
            if best is None: break
            l,r=best; merged=[]; j=0
            while j<len(symbols):
                if j+1<len(symbols) and symbols[j]==l and symbols[j+1]==r:
                    merged.append(l+r); j+=2
                else:
                    merged.append(symbols[j]); j+=1
            symbols=merged
        if len(self._word_cache)>=250000: self._word_cache.clear()
        self._word_cache[word]=tuple(symbols)
        return symbols
    def encode(self,text):
        out=[]; unk=self.unk_token_id
        for unit in re.findall(r"\S+|\s+",str(text)):
            for s in self.encode_word(unit): out.append(self.stoi.get(s,unk))
        return out
    def decode(self,ids):
        buf=bytearray()
        for idx in ids:
            tok=self.itos[int(idx)]
            if tok not in BPE_SPECIALS: buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8",errors="replace")
    @classmethod
    def load_json(cls,path):
        with open(path,encoding="utf-8") as f:
            obj=json.load(f)
        tok=cls(obj["vocab_size"])
        tok.merges=[list(x) for x in obj["merges"]]
        tok.itos=list(obj["itos"]); tok.stoi={t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size=len(tok.itos); tok.symbol_bytes={}
        for b in range(256):
            s=f"{b:02x}"; tok.symbol_bytes[s]=bytes([b])
        for l,r in tok.merges: tok.symbol_bytes[l+r]=tok.symbol_bytes[l]+tok.symbol_bytes[r]
        tok._reset_encode_cache(); return tok

tokenizer=LegalBPE.load_json(TOKENIZER_FILE)
assert len(tokenizer)==VOCAB_SIZE
probe="IN THE SUPREME COURT OF INDIA\nSection 14: ₹100 — legal test."
assert tokenizer.decode(tokenizer.encode(probe))==probe
print("Tokenizer PASS:",len(tokenizer))


Tokenizer PASS: 24576


In [4]:
# CELL 4 — download Phase4 + build leakage-safe splits (+ normalisation)
FNS={
    "train":"Phase_4_CJPE/Train/train.json",
    "dev":"Phase_4_CJPE/Train/dev.json",
    "test":"Phase_4_CJPE/Test/test.json",
}
paths={k:hf_hub_download(repo_id=HF_REPO,repo_type="dataset",filename=v,token=HF_TOKEN) for k,v in FNS.items()}
DATA_SHA256={k:sha256_file(v) for k,v in paths.items()}
print("dataset SHA256:")
for k,h in DATA_SHA256.items(): print(f"  {k}: {h}")

train_raw=read_json(paths["train"])
dev_raw=read_json(paths["dev"])
test_raw=read_json(paths["test"])
RAW_ROW_COUNTS={"train":len(train_raw),"dev":len(dev_raw),"test":len(test_raw)}

def dedup_labeled(rows,name):
    seen={}; conflicts=set(); text_diff=0
    for r in rows:
        cid=str(r["id"]); y=int(r["label"]); txt=str(r["text"])
        assert y in (0,1)
        if cid in seen:
            if seen[cid]["label"]!=y:
                conflicts.add(cid)
            elif seen[cid]["text"]!=txt:
                text_diff+=1
                if len(txt)>len(seen[cid]["text"]):
                    seen[cid]["text"]=txt
        else:
            seen[cid]={"id":cid,"text":txt,"label":y}
    for cid in conflicts: seen.pop(cid,None)
    print(
        f"{name}: rows={len(rows)} unique={len(seen)+len(conflicts)} | "
        f"same-ID different-text={text_diff} | label-conflict IDs dropped={len(conflicts)}"
    )
    return list(seen.values())

train_records=dedup_labeled(train_raw,"train")
dev_unique=dedup_labeled(dev_raw,"dev")

# TEST: copy ONLY id/text. Never read/use test label or expert fields.
test_records=[{"id":str(r["id"]),"text":str(r["text"])} for r in test_raw]
test_ids={r["id"] for r in test_records}
assert len(test_ids)==len(test_records),f"Duplicate TEST IDs: {len(test_records)-len(test_ids)}"

overlap_ids={r["id"] for r in dev_unique}&test_ids
overlap_count=len(overlap_ids)
clean_dev=[r for r in dev_unique if r["id"] not in test_ids]

if len(train_records)!=32305:
    print("⚠️ TRAIN unique count differs from earlier audit:",len(train_records))
assert len(test_records)==1517,f"Unexpected TEST size: {len(test_records)}"
assert len(clean_dev)==994,f"Unexpected CLEAN DEV size: {len(clean_dev)}"

# Both known hosted snapshots are valid:
# old DEV: 2511 unique, 1517 TEST overlap -> 994 clean
# new DEV:  994 unique, already clean -> 0 TEST overlap
if overlap_count==1517:
    print("✅ Raw DEV contains 1517 TEST-overlap cases; excluded them.")
elif overlap_count==0 and len(dev_unique)==994:
    print("✅ DEV is already leakage-clean; no TEST-overlap cases are present.")
else:
    print("⚠️ Unexpected DEV layout:",
          f"dev_unique={len(dev_unique)} overlap={overlap_count} clean_dev={len(clean_dev)}")

train_ids={r["id"] for r in train_records}
clean_dev_ids={r["id"] for r in clean_dev}
assert not (train_ids&clean_dev_ids),f"TRAIN↔CLEAN-DEV overlap: {len(train_ids&clean_dev_ids)}"
assert not (train_ids&test_ids),f"TRAIN↔TEST overlap: {len(train_ids&test_ids)}"
assert not (clean_dev_ids&test_ids),f"CLEAN DEV↔TEST overlap: {len(clean_dev_ids&test_ids)}"

def upper_ratio(t):
    t=t[:20000]
    letters=sum(c.isalpha() for c in t); up=sum(c.isupper() for c in t)
    return up/max(letters,1)

_rng=random.Random(SEED)
def sample_ratio(rows,k=1500):
    idx=_rng.sample(range(len(rows)),min(k,len(rows)))
    return np.array([upper_ratio(rows[i]["text"]) for i in idx])

ur_tr,ur_dv,ur_te=sample_ratio(train_records),sample_ratio(clean_dev),sample_ratio(test_records)
print(f"uppercase-letter ratio train median={np.median(ur_tr):.4f} "
      f"(>2% in {100*np.mean(ur_tr>0.02):.1f}% docs) | "
      f"clean-dev median={np.median(ur_dv):.4f} | test median={np.median(ur_te):.4f}")
print("newline-containing docs train: {:.1f}% | dev: {:.1f}% | test: {:.1f}%".format(
    *[100*np.mean(["\n" in r["text"] for r in rows[:2000]])
      for rows in (train_records,clean_dev,test_records)]))

LOWERCASE=(CASE_MODE=="lower") or (CASE_MODE=="auto" and float(np.median(ur_te))<0.01)
print("LOWERCASE:",LOWERCASE,"| COLLAPSE_WS:",COLLAPSE_WS)

def normalize_text(t):
    if LOWERCASE: t=t.lower()
    if COLLAPSE_WS: t=re.sub(r"\s+"," ",t).strip()
    return t

for rows in (train_records,clean_dev,test_records):
    for r in rows: r["text"]=normalize_text(r["text"])

del train_raw,dev_raw,test_raw,dev_unique
gc.collect()

print("TRAIN:",len(train_records),Counter(r["label"] for r in train_records))
print("CLEAN DEV:",len(clean_dev),Counter(r["label"] for r in clean_dev))
print("TEST:",len(test_records))
print("Excluded DEV↔TEST overlap:",overlap_count)
print("✅ TEST label/expert fields ignored.")


Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

dataset SHA256:
  train: a115cc722e9de8b68ac5d41179c12d86c2d537c0de8ff16412c44c6c8d1cce72
  dev: aa2d80ae77f5d532fbc3f30e4dea2776700b8e92e71c603737d8f41a9d30b514
  test: f3498fb832e90fe7264fcb6770918eb5261409e9a952d47a2979b6ed5b153b59
train: rows=37387 unique=32305 | same-ID different-text=0 | label-conflict IDs dropped=0
dev: rows=1988 unique=994 | same-ID different-text=0 | label-conflict IDs dropped=0
✅ DEV is already leakage-clean; no TEST-overlap cases are present.
uppercase-letter ratio train median=0.0351 (>2% in 97.1% docs) | clean-dev median=0.0000 | test median=0.0000
newline-containing docs train: 0.0% | dev: 100.0% | test: 98.9%
LOWERCASE: True | COLLAPSE_WS: True
TRAIN: 32305 Counter({0: 18920, 1: 13385})
CLEAN DEV: 994 Counter({0: 497, 1: 497})
TEST: 1517
Excluded DEV↔TEST overlap: 0
✅ TEST label/expert fields ignored.


In [5]:
# CELL 5 — exact Phase-1 Model3 architecture + strict checkpoint load
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self,d_model,max_positions):
        super().__init__()
        pos=torch.arange(max_positions,dtype=torch.float32).unsqueeze(1)
        div=torch.exp(torch.arange(0,d_model,2,dtype=torch.float32)*(-math.log(10000.0)/d_model))
        pe=torch.zeros(max_positions,d_model,dtype=torch.float32)
        pe[:,0::2]=torch.sin(pos*div)
        pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer("pe",pe,persistent=False)
    def forward(self,position_ids):
        return self.pe[position_ids]

class EncoderSelfAttention(nn.Module):
    def __init__(self,d_model,n_heads,dropout):
        super().__init__(); assert d_model%n_heads==0
        self.n_heads=n_heads; self.head_dim=d_model//n_heads
        self.qkv=nn.Linear(d_model,3*d_model)
        self.out_proj=nn.Linear(d_model,d_model)
        self.dropout=dropout
    def forward(self,x,key_valid=None):
        B,T,C=x.shape
        qkv=self.qkv(x).view(B,T,3,self.n_heads,self.head_dim)
        q,k,v=qkv.unbind(dim=2)
        q=q.transpose(1,2); k=k.transpose(1,2); v=v.transpose(1,2)
        mask=None if key_valid is None else key_valid[:,None,None,:].bool()
        y=F.scaled_dot_product_attention(
            q,k,v,attn_mask=mask,
            dropout_p=self.dropout if self.training else 0.0,is_causal=False
        )
        y=y.transpose(1,2).contiguous().view(B,T,C)
        return self.out_proj(y)

class DecoderSelfAttention(nn.Module):
    def __init__(self,d_model,n_heads,dropout):
        super().__init__(); assert d_model%n_heads==0
        self.n_heads=n_heads; self.head_dim=d_model//n_heads
        self.qkv=nn.Linear(d_model,3*d_model)
        self.out_proj=nn.Linear(d_model,d_model)
        self.dropout=dropout
    def forward(self,x,past_kv=None,use_cache=False):
        B,T,C=x.shape
        qkv=self.qkv(x).view(B,T,3,self.n_heads,self.head_dim)
        q,k,v=qkv.unbind(dim=2)
        q=q.transpose(1,2); k=k.transpose(1,2); v=v.transpose(1,2)
        mask,is_causal=None,True
        if past_kv is not None:
            pk,pv=past_kv
            k=torch.cat([pk,k],2); v=torch.cat([pv,v],2)
            if T==1:
                is_causal=False
            else:
                S=k.size(2); P=S-T
                mask=torch.ones(T,S,device=x.device).tril(diagonal=P).bool()
                is_causal=False
        y=F.scaled_dot_product_attention(
            q,k,v,attn_mask=mask,
            dropout_p=self.dropout if self.training else 0.0,is_causal=is_causal
        )
        y=y.transpose(1,2).contiguous().view(B,T,C)
        return self.out_proj(y),((k,v) if use_cache else None)

class CrossAttention(nn.Module):
    def __init__(self,d_model,n_heads,dropout):
        super().__init__(); assert d_model%n_heads==0
        self.n_heads=n_heads; self.head_dim=d_model//n_heads
        self.q_proj=nn.Linear(d_model,d_model)
        self.kv_proj=nn.Linear(d_model,2*d_model)
        self.out_proj=nn.Linear(d_model,d_model)
        self.dropout=dropout
    def forward(self,x,memory,memory_valid=None,cached_kv=None,use_cache=False):
        B,T,C=x.shape
        q=self.q_proj(x).view(B,T,self.n_heads,self.head_dim).transpose(1,2)
        if cached_kv is None:
            S=memory.size(1)
            kv=self.kv_proj(memory).view(B,S,2,self.n_heads,self.head_dim)
            k,v=kv.unbind(dim=2)
            k=k.transpose(1,2); v=v.transpose(1,2)
        else:
            k,v=cached_kv
        mask=None if memory_valid is None else memory_valid[:,None,None,:].bool()
        y=F.scaled_dot_product_attention(
            q,k,v,attn_mask=mask,
            dropout_p=self.dropout if self.training else 0.0,is_causal=False
        )
        y=y.transpose(1,2).contiguous().view(B,T,C)
        y=self.out_proj(y)
        return y,((k,v) if use_cache else None)

class EncoderBlock(nn.Module):
    def __init__(self,d_model,n_heads,ffn_dim,dropout):
        super().__init__()
        self.ln1=nn.LayerNorm(d_model)
        self.attn=EncoderSelfAttention(d_model,n_heads,dropout)
        self.ln2=nn.LayerNorm(d_model)
        self.ff=nn.Sequential(
            nn.Linear(d_model,ffn_dim),nn.GELU(),
            nn.Linear(ffn_dim,d_model),nn.Dropout(dropout)
        )
        self.drop=nn.Dropout(dropout)
    def forward(self,x,key_valid=None):
        x=x+self.drop(self.attn(self.ln1(x),key_valid=key_valid))
        x=x+self.ff(self.ln2(x))
        return x

class DecoderBlock(nn.Module):
    def __init__(self,d_model,n_heads,ffn_dim,dropout):
        super().__init__()
        self.ln1=nn.LayerNorm(d_model)
        self.self_attn=DecoderSelfAttention(d_model,n_heads,dropout)
        self.ln2=nn.LayerNorm(d_model)
        self.cross_attn=CrossAttention(d_model,n_heads,dropout)
        self.ln3=nn.LayerNorm(d_model)
        self.ff=nn.Sequential(
            nn.Linear(d_model,ffn_dim),nn.GELU(),
            nn.Linear(ffn_dim,d_model),nn.Dropout(dropout)
        )
        self.drop=nn.Dropout(dropout)
    def forward(self,x,memory,memory_valid=None,past_self_kv=None,cached_cross_kv=None,use_cache=False):
        a,new_self=self.self_attn(self.ln1(x),past_kv=past_self_kv,use_cache=use_cache)
        x=x+self.drop(a)
        c,new_cross=self.cross_attn(
            self.ln2(x),memory,memory_valid=memory_valid,
            cached_kv=cached_cross_kv,use_cache=use_cache
        )
        x=x+self.drop(c)
        x=x+self.ff(self.ln3(x))
        return x,new_self,new_cross

class EncoderDecoderLM(nn.Module):
    def __init__(self,vocab_size,d_model,n_encoder_layers,n_decoder_layers,n_heads,ffn_dim,
                 dropout,max_encoder_positions,max_decoder_positions):
        super().__init__()
        self.vocab_size=vocab_size; self.d_model=d_model
        self.emb_scale=math.sqrt(d_model)
        self.max_encoder_positions=max_encoder_positions
        self.max_decoder_positions=max_decoder_positions
        self.token_emb=nn.Embedding(vocab_size,d_model)
        self.enc_pos=SinusoidalPositionalEncoding(d_model,max_encoder_positions)
        self.dec_pos=SinusoidalPositionalEncoding(d_model,max_decoder_positions)
        self.drop=nn.Dropout(dropout)
        self.encoder=nn.ModuleList([
            EncoderBlock(d_model,n_heads,ffn_dim,dropout) for _ in range(n_encoder_layers)
        ])
        self.decoder=nn.ModuleList([
            DecoderBlock(d_model,n_heads,ffn_dim,dropout) for _ in range(n_decoder_layers)
        ])
        self.enc_ln=nn.LayerNorm(d_model)
        self.dec_ln=nn.LayerNorm(d_model)
        self.apply(self._init_weights)
        enc_scaled=0.02/math.sqrt(2*n_encoder_layers)
        dec_scaled=0.02/math.sqrt(3*n_decoder_layers)
        for b in self.encoder:
            nn.init.normal_(b.attn.out_proj.weight,mean=0.0,std=enc_scaled)
            nn.init.normal_(b.ff[2].weight,mean=0.0,std=enc_scaled)
        for b in self.decoder:
            nn.init.normal_(b.self_attn.out_proj.weight,mean=0.0,std=dec_scaled)
            nn.init.normal_(b.cross_attn.out_proj.weight,mean=0.0,std=dec_scaled)
            nn.init.normal_(b.ff[2].weight,mean=0.0,std=dec_scaled)

    def _init_weights(self,m):
        if isinstance(m,nn.Linear):
            nn.init.normal_(m.weight,mean=0.0,std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m,nn.Embedding):
            nn.init.normal_(m.weight,mean=0.0,std=0.02)

    def encode(self,input_ids,key_valid=None,position_ids=None):
        B,S=input_ids.shape
        if S>self.max_encoder_positions:
            raise ValueError(f"encoder length {S} > {self.max_encoder_positions}")
        if position_ids is None:
            if key_valid is None:
                position_ids=torch.arange(S,device=input_ids.device)[None,:].expand(B,S)
            else:
                position_ids=(key_valid.long().cumsum(1)-1).clamp(min=0)
        x=self.token_emb(input_ids)*self.emb_scale + self.enc_pos(position_ids).to(self.token_emb.weight.dtype)
        x=self.drop(x)
        for block in self.encoder:
            x=block(x,key_valid=key_valid)
        return self.enc_ln(x)

    def decode_hidden(self,decoder_ids,memory,memory_valid=None,past_self_key_values=None,
                      cross_key_values=None,use_cache=False,position_ids=None):
        B,T=decoder_ids.shape
        past_len=0 if past_self_key_values is None else past_self_key_values[0][0].size(2)
        if past_len+T>self.max_decoder_positions:
            raise ValueError(f"decoder length {past_len+T} > {self.max_decoder_positions}")
        if position_ids is None:
            position_ids=torch.arange(past_len,past_len+T,device=decoder_ids.device)[None,:].expand(B,T)
        x=self.token_emb(decoder_ids)*self.emb_scale + self.dec_pos(position_ids).to(self.token_emb.weight.dtype)
        x=self.drop(x)
        ns,nc=([] if use_cache else None),([] if use_cache else None)
        for i,block in enumerate(self.decoder):
            ps=None if past_self_key_values is None else past_self_key_values[i]
            ck=None if cross_key_values is None else cross_key_values[i]
            x,sk,ckv=block(
                x,memory,memory_valid=memory_valid,
                past_self_kv=ps,cached_cross_kv=ck,use_cache=use_cache
            )
            if use_cache:
                ns.append(sk); nc.append(ckv)
        return self.dec_ln(x),ns,nc

    def project(self,hidden):
        return F.linear(hidden,self.token_emb.weight)

def build_backbone():
    return EncoderDecoderLM(
        VOCAB_SIZE,D_MODEL,N_ENCODER_LAYERS,N_DECODER_LAYERS,N_HEADS,FFN_DIM,
        DROPOUT,MAX_ENCODER_POSITIONS,MAX_DECODER_POSITIONS
    )

# Strict load BEFORE adapter injection.
try:
    phase1_obj=torch.load(PHASE1_CKPT,map_location="cpu",weights_only=True)
except Exception:
    phase1_obj=torch.load(PHASE1_CKPT,map_location="cpu",weights_only=False)

assert isinstance(phase1_obj,dict) and "model" in phase1_obj,"Model3 checkpoint must contain key 'model'."
cfg=phase1_obj.get("config",{})
print("checkpoint step:",phase1_obj.get("global_step"))
print("checkpoint val:",phase1_obj.get("best_val",phase1_obj.get("val_loss")))
print("checkpoint config:",cfg)

# Fail early if this is not the expected Model3 family.
for key,expected in [
    ("vocab_size",VOCAB_SIZE),("d_model",D_MODEL),("n_encoder_layers",N_ENCODER_LAYERS),
    ("n_decoder_layers",N_DECODER_LAYERS),("n_heads",N_HEADS),("ffn_dim",FFN_DIM)
]:
    if key in cfg:
        assert int(cfg[key])==expected,(key,cfg[key],expected)

assert len(tokenizer)==VOCAB_SIZE,(len(tokenizer),VOCAB_SIZE)
backbone=build_backbone()
backbone.load_state_dict(phase1_obj["model"],strict=True)

n_params=sum(p.numel() for p in backbone.parameters())
assert 75_000_000<=n_params<=85_000_000,n_params
print(f"✅ Strict Phase-1 Model3 load PASS | params={n_params/1e6:.3f}M")


checkpoint step: 16266
checkpoint val: 1.885548301174893
checkpoint config: {'architecture': 'encoder_decoder_transformer', 'vocab_size': 24576, 'd_model': 512, 'n_encoder_layers': 9, 'n_decoder_layers': 9, 'n_heads': 8, 'ffn_dim': 2048, 'dropout': 0.0, 'max_encoder_positions': 2048, 'max_decoder_positions': 1536, 'seq_len': 1536, 'prompt_p10_tokens': 1050, 'prompt_p95_tokens': 1139, 'expected_cont_tokens': 384, 'prompt_split_min': 1050, 'prompt_split_max': 1139}
✅ Strict Phase-1 Model3 load PASS | params=78.793M


In [6]:
# CELL 6 — true post-residual Houlsby adapters + gated encoder/decoder hierarchical head
class BottleneckAdapter(nn.Module):
    """Post-residual Houlsby adapter: u + Up(Dropout(GELU(Down(LN(u)))))."""
    def __init__(self,d_model,bottleneck,dropout):
        super().__init__()
        self.ln=nn.LayerNorm(d_model)
        self.down=nn.Linear(d_model,bottleneck)
        self.act=nn.GELU()
        self.drop=nn.Dropout(dropout)
        self.up=nn.Linear(bottleneck,d_model)
        nn.init.ones_(self.ln.weight); nn.init.zeros_(self.ln.bias)
        nn.init.normal_(self.down.weight,mean=0.0,std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self,u):
        in_dtype=u.dtype
        with torch.autocast(device_type=u.device.type if u.device.type in ("cuda","cpu") else "cuda",enabled=False):
            z=self.ln(u.float()); z=self.down(z); z=self.act(z); z=self.drop(z); z=self.up(z)
        return u+z.to(dtype=in_dtype)

class EncoderBlockWithAdapters(nn.Module):
    def __init__(self,base,d_model,bottleneck,dropout):
        super().__init__(); self.base=base
        self.attn_adapter=BottleneckAdapter(d_model,bottleneck,dropout)
        self.ffn_adapter=BottleneckAdapter(d_model,bottleneck,dropout)
    def forward(self,x,key_valid=None):
        a=self.base.drop(self.base.attn(self.base.ln1(x),key_valid=key_valid))
        x=x+a; x=self.attn_adapter(x)
        f=self.base.ff(self.base.ln2(x))
        x=x+f; x=self.ffn_adapter(x)
        return x

class DecoderBlockWithAdapters(nn.Module):
    def __init__(self,base,d_model,bottleneck,dropout):
        super().__init__(); self.base=base
        self.self_adapter=BottleneckAdapter(d_model,bottleneck,dropout)
        self.cross_adapter=BottleneckAdapter(d_model,bottleneck,dropout)
        self.ffn_adapter=BottleneckAdapter(d_model,bottleneck,dropout)
    def forward(self,x,memory,memory_valid=None,past_self_kv=None,cached_cross_kv=None,use_cache=False):
        a,new_self=self.base.self_attn(self.base.ln1(x),past_kv=past_self_kv,use_cache=use_cache)
        x=x+self.base.drop(a); x=self.self_adapter(x)
        c,new_cross=self.base.cross_attn(self.base.ln2(x),memory,memory_valid=memory_valid,cached_kv=cached_cross_kv,use_cache=use_cache)
        x=x+self.base.drop(c); x=self.cross_adapter(x)
        f=self.base.ff(self.base.ln3(x))
        x=x+f; x=self.ffn_adapter(x)
        return x,new_self,new_cross

def freeze_backbone(b):
    for p in b.parameters(): p.requires_grad=False

def inject_adapters(b):
    freeze_backbone(b)
    b.encoder=nn.ModuleList([EncoderBlockWithAdapters(x,b.d_model,ADAPTER_BOTTLENECK,ADAPTER_DROPOUT) for x in b.encoder])
    b.decoder=nn.ModuleList([DecoderBlockWithAdapters(x,b.d_model,ADAPTER_BOTTLENECK,ADAPTER_DROPOUT) for x in b.decoder])
    return 2*len(b.encoder)+3*len(b.decoder)

# Exact identity check before/after zero-init adapter injection.
backbone.eval()
_sanity=torch.randint(4,VOCAB_SIZE,(2,48),generator=torch.Generator().manual_seed(SEED))
_smask=torch.ones_like(_sanity,dtype=torch.bool)
_dbos=torch.full((2,1),int(tokenizer.bos_token_id),dtype=torch.long)
with torch.no_grad():
    _m0=backbone.encode(_sanity,key_valid=_smask); _d0,_,_=backbone.decode_hidden(_dbos,_m0,memory_valid=_smask)
N_ADAPTERS=inject_adapters(backbone)
backbone.eval()
with torch.no_grad():
    _m1=backbone.encode(_sanity,key_valid=_smask); _d1,_,_=backbone.decode_hidden(_dbos,_m1,memory_valid=_smask)
_identity_err=max(float((_m0-_m1).abs().max()),float((_d0-_d1).abs().max()))
print("Adapters injected:",N_ADAPTERS); print("Adapter identity max |Δ hidden|:",_identity_err)
assert _identity_err<1e-6,_identity_err
del _m0,_m1,_d0,_d1,_sanity,_smask,_dbos

class HierarchicalCJPE(nn.Module):
    def __init__(self,backbone):
        super().__init__(); self.backbone=backbone
        self.decoder_gate_logit=nn.Parameter(torch.tensor(-2.0))
        self.chunk_proj=nn.Sequential(
            nn.Linear(4*D_MODEL,CHUNK_PROJ_DIM),nn.GELU(),nn.LayerNorm(CHUNK_PROJ_DIM),nn.Dropout(HEAD_DROPOUT)
        )
        self.chunk_pos=nn.Embedding(N_CHUNKS,CHUNK_PROJ_DIM)
        self.attn_score=nn.Sequential(nn.Linear(CHUNK_PROJ_DIM,128),nn.Tanh(),nn.Linear(128,1))
        self.fc=nn.Sequential(nn.Linear(2*CHUNK_PROJ_DIM,256),nn.GELU(),nn.Dropout(HEAD_DROPOUT),nn.Linear(256,2))
    def forward(self,input_ids,valid):
        B,K,T=input_ids.shape
        x=input_ids.view(B*K,T); m=valid.view(B*K,T)
        memory=self.backbone.encode(x,key_valid=m)
        mf=m.unsqueeze(-1).to(memory.dtype)
        mean=(memory*mf).sum(1)/mf.sum(1).clamp(min=1)
        last_idx=m.long().sum(1).clamp(min=1)-1
        last=memory[torch.arange(B*K,device=memory.device),last_idx]
        neg=torch.finfo(memory.dtype).min
        max_pool=memory.masked_fill(~m.unsqueeze(-1),neg).max(1).values
        max_pool=torch.where(torch.isfinite(max_pool),max_pool,torch.zeros_like(max_pool))
        bos=torch.full((B*K,1),int(tokenizer.bos_token_id),dtype=torch.long,device=x.device)
        dh,_,_=self.backbone.decode_hidden(bos,memory,memory_valid=m)
        gate=torch.sigmoid(self.decoder_gate_logit)
        dec_summary=gate*dh[:,0]
        z=self.chunk_proj(torch.cat([last.float(),mean.float(),max_pool.float(),dec_summary.float()],dim=-1)).view(B,K,-1)
        z=z+self.chunk_pos(torch.arange(K,device=z.device)[None,:])
        w=torch.softmax(self.attn_score(z).squeeze(-1),dim=-1)
        att=(z*w.unsqueeze(-1)).sum(1); mx=z.max(1).values
        return self.fc(torch.cat([att,mx],dim=-1)),w

model=HierarchicalCJPE(backbone).to(DEVICE)
trainable_names=[n for n,p in model.named_parameters() if p.requires_grad]
adapter_names=[n for n in trainable_names if "adapter." in n]
assert N_ADAPTERS==45,N_ADAPTERS; assert adapter_names
_ALLOWED_HEAD_PREFIXES=("decoder_gate_logit","chunk_proj.","chunk_pos.","attn_score.","fc.")
assert all(("adapter." in n) or n.startswith(_ALLOWED_HEAD_PREFIXES) for n in trainable_names),trainable_names[:20]
adapter_params=sum(p.numel() for n,p in model.named_parameters() if p.requires_grad and "adapter." in n)
head_params=sum(p.numel() for n,p in model.named_parameters() if p.requires_grad and "adapter." not in n)
total_params=sum(p.numel() for p in model.parameters())
print(f"Adapter trainable={adapter_params/1e6:.3f}M | head={head_params/1e6:.3f}M | total={total_params/1e6:.3f}M")
print(f"Total trainable={(adapter_params+head_params)/1e6:.3f}M ({100*(adapter_params+head_params)/total_params:.2f}% of model)")
print("Initial decoder gate:",float(torch.sigmoid(model.decoder_gate_logit).detach().cpu()))
print("✅ All original Phase-1 weights are frozen.")


Adapters injected: 45
Adapter identity max |Δ hidden|: 0.0
Adapter trainable=5.973M | head=1.036M | total=85.802M
Total trainable=7.009M (8.17% of model)
Initial decoder gate: 0.11920291930437088
✅ All original Phase-1 weights are frozen.


In [7]:
# CELL 7 — tokenize/select HEAD-MIDDLE-EVIDENCE-TAIL
SLICE_CHARS=CHUNK_LEN*8     # ~3k chars >> 384 BPE tokens

def _snap_forward(text,i):
    i=max(0,min(int(i),max(0,len(text)-1)))
    m=re.search(r"\s",text[i:i+400])
    return i+m.end() if m else i

def _enc(t):
    return tokenizer.encode(t)

def _window_start(text,center,width):
    s=max(0,int(center)-width//2)
    return _snap_forward(text,s)

def _last_evidence_position(text):
    low=text.lower()
    best=-1
    for cue in EVIDENCE_CUES:
        if cue in WEAK_CUES: continue
        p=low.rfind(cue)
        if p>best: best=p
    if best<0:   # fallback to weaker cues only if no explicit disposition found
        for cue in WEAK_CUES:
            p=low.rfind(cue)
            if p>best: best=p
    return best

SELECT_STATS=Counter()   # diagnostics only

def select_chunks(text):
    """
    Four views of a long case:
      head / middle / evidence-or-late / tail.
    For short cases we derive all views from exact full BPE tokens.
    """
    n=len(text); W=SLICE_CHARS

    if n<=4*W:
        SELECT_STATS["short_doc"]+=1
        ids=_enc(text) or [tokenizer.unk_token_id]
        L=len(ids)

        if L<=CHUNK_LEN:
            ch=[ids,ids,ids,ids]
        else:
            def tok_window(frac):
                center=int(round(frac*(L-1)))
                s=max(0,min(L-CHUNK_LEN,center-CHUNK_LEN//2))
                return ids[s:s+CHUNK_LEN]

            # evidence view falls back to a late (75%) view for short docs.
            ch=[
                ids[:CHUNK_LEN],
                tok_window(0.50),
                tok_window(0.75),
                ids[-CHUNK_LEN:],
            ]
        est=L

    else:
        # Prefix
        head=_enc(text[:W])

        # True character-centred middle slice.
        ms=_window_start(text,n*0.50,W)
        mid=_enc(text[ms:ms+W])

        # Evidence chunk: centre around the LAST neutral disposition cue.
        # If absent, use a late-document (75%) region.
        ep=_last_evidence_position(text)
        SELECT_STATS["long_doc"]+=1; SELECT_STATS["cue_found" if ep>=0 else "cue_missing"]+=1
        ec=(ep if ep>=0 else int(n*0.75))
        es=_window_start(text,ec,W)
        evidence=_enc(text[es:es+W])

        # Suffix
        ts=_snap_forward(text,max(0,n-W))
        tail=_enc(text[ts:])

        if min(len(head),len(mid),len(evidence),len(tail))<CHUNK_LEN:
            # Rare token-density/pathological fallback: exact full tokenisation.
            SELECT_STATS["fallback_full_tokenisation"]+=1
            ids=_enc(text) or [tokenizer.unk_token_id]
            L=len(ids)
            def tok_window(frac):
                center=int(round(frac*(L-1)))
                s=max(0,min(max(0,L-CHUNK_LEN),center-CHUNK_LEN//2))
                return ids[s:s+CHUNK_LEN]
            ch=[
                ids[:CHUNK_LEN],
                tok_window(0.50),
                tok_window(0.75),
                ids[-CHUNK_LEN:],
            ]
            est=L
        else:
            ch=[
                head[:CHUNK_LEN],
                mid[:CHUNK_LEN],
                evidence[:CHUNK_LEN],
                tail[-CHUNK_LEN:],
            ]
            est=int(n*len(head)/max(W,1))  # statistics only

    return [np.asarray(x,dtype=np.uint16) for x in ch],est

class CaseDataset(Dataset):
    def __init__(self,records,labeled):
        self.records=records; self.labeled=labeled; self.chunks=[]; self.full_lens=[]
        SELECT_STATS.clear(); t0=time.time()
        for i,r in enumerate(records):
            ch,L=select_chunks(r["text"]); self.chunks.append(ch); self.full_lens.append(L)
            if (i+1)%5000==0: print(f"tokenized {i+1}/{len(records)} elapsed={time.time()-t0:.0f}s",flush=True)
        print(f"done {len(records)} in {time.time()-t0:.0f}s | est. median BPE={np.median(self.full_lens):.0f} p95={np.percentile(self.full_lens,95):.0f}")
        print("   chunk-selection stats:",dict(SELECT_STATS))
    def __len__(self): return len(self.records)
    def __getitem__(self,i):
        r=self.records[i]; d={"id":r["id"],"chunks":self.chunks[i]}
        if self.labeled: d["label"]=int(r["label"])
        return d

def collate_cases(batch):
    B=len(batch)
    ids=torch.full((B,N_CHUNKS,CHUNK_LEN),tokenizer.pad_token_id,dtype=torch.long)
    valid=torch.zeros((B,N_CHUNKS,CHUNK_LEN),dtype=torch.bool)
    for b,x in enumerate(batch):
        for k,c in enumerate(x["chunks"]):
            z=torch.from_numpy(c.astype(np.int64)); n=min(len(z),CHUNK_LEN)
            ids[b,k,:n]=z[:n]; valid[b,k,:n]=True
    out={"input_ids":ids,"valid":valid,"ids":[x["id"] for x in batch]}
    if "label" in batch[0]:
        out["labels"]=torch.tensor([x["label"] for x in batch],dtype=torch.long)
    return out

train_ds=CaseDataset(train_records,True)
dev_ds=CaseDataset(clean_dev,True)
test_ds=CaseDataset(test_records,False)

g=torch.Generator(); g.manual_seed(SEED)
_kw=dict(collate_fn=collate_cases,num_workers=NUM_WORKERS,pin_memory=DEVICE.type=="cuda",persistent_workers=NUM_WORKERS>0)
train_loader=DataLoader(train_ds,batch_size=MICRO_BATCH_SIZE,shuffle=True,generator=g,**_kw)
dev_loader=DataLoader(dev_ds,batch_size=EVAL_BATCH_SIZE,shuffle=False,**_kw)
test_loader=DataLoader(test_ds,batch_size=EVAL_BATCH_SIZE,shuffle=False,**_kw)

print("batches:",len(train_loader),len(dev_loader),len(test_loader))

# ---- Model3-specific route sanity (encoder -> decoder cross-attention summary -> head) ----
@torch.no_grad()
def model3_route_sanity():
    was_training = model.training
    model.eval()

    b = next(iter(dev_loader))
    x = b["input_ids"][:1].to(DEVICE)
    v = b["valid"][:1].to(DEVICE)

    with torch.amp.autocast(
        "cuda",
        dtype=torch.float16,
        enabled=(DEVICE.type=="cuda" and USE_AMP),
    ):
        lg, a = model(x, v)

    assert tuple(lg.shape) == (1, 2), lg.shape
    assert tuple(a.shape) == (1, N_CHUNKS), a.shape
    assert bool(torch.isfinite(lg).all()), "Non-finite classifier logits"
    assert bool(torch.isfinite(a).all()), "Non-finite chunk attention weights"
    assert torch.allclose(
        a.sum(-1),
        torch.ones(1, device=a.device, dtype=a.dtype),
        atol=2e-3,
    ), a.sum(-1)

    print("✅ Model3 encoder→decoder-summary→hierarchical-head route PASS")
    model.train(was_training)

model3_route_sanity()


# ---- Phase-1 Model3 tokenizer/checkpoint sanity ----
# Recreates a small version of the original encoder-decoder LM objective.
@torch.no_grad()
def phase1_seq2seq_sanity(n_docs=12):
    was_training = model.training
    model.eval()

    losses = []
    PRIME_TOKENS = 32

    for i in range(min(n_docs, len(train_records))):
        ids = tokenizer.encode(train_records[i]["text"][:12000])

        if len(ids) < 180:
            continue

        ids = ids[:256]
        split = min(160, len(ids) - 48)

        prefix_ids = ids[:split]
        target_ids = ids[split:min(len(ids), split + 64)]

        if len(target_ids) < 24:
            continue

        # Encoder input = source legal context.
        enc_ids = torch.tensor(
            [prefix_ids],
            dtype=torch.long,
            device=DEVICE,
        )
        enc_valid = torch.ones_like(enc_ids, dtype=torch.bool)

        # Decoder gets BOS + a short priming suffix + target[:-1].
        primed = prefix_ids[-min(PRIME_TOKENS, len(prefix_ids)):]
        dec_in = [int(tokenizer.bos_token_id)] + primed + target_ids[:-1]

        dec_ids = torch.tensor(
            [dec_in],
            dtype=torch.long,
            device=DEVICE,
        )

        with torch.amp.autocast(
            "cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type=="cuda" and USE_AMP),
        ):
            memory = model.backbone.encode(
                enc_ids,
                key_valid=enc_valid,
            )

            h, _, _ = model.backbone.decode_hidden(
                dec_ids,
                memory,
                memory_valid=enc_valid,
            )

            # Skip BOS + priming positions and score only the true target.
            skip = len(primed)

            logits = model.backbone.project(
                h[:, skip:skip + len(target_ids)]
            ).float()

        y = torch.tensor(
            [target_ids],
            dtype=torch.long,
            device=DEVICE,
        )

        assert logits.size(1) == y.size(1), (logits.shape, y.shape)

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            y.reshape(-1),
        )
        losses.append(float(loss))

    model.train(was_training)

    if not losses:
        print("⚠️ Phase-1 Model3 seq2seq sanity skipped (sampled docs too short).")
        return

    mean_loss = float(np.mean(losses))

    print(
        f"Phase-1 Model3 seq2seq sanity: "
        f"loss={mean_loss:.3f} | "
        f"ppl={math.exp(min(mean_loss,20)):.1f} | "
        f"chance ln(V)={math.log(VOCAB_SIZE):.2f}"
    )

    if mean_loss > 7.0:
        print(
            "⚠️ Phase-1 loss is unusually high. "
            "Verify that the selected tokenizer belongs to this Model3 checkpoint."
        )

    assert mean_loss < 8.5, (
        f"Phase-1 Model3 sanity FAILED: mean loss={mean_loss:.3f}. "
        "Most likely a tokenizer/checkpoint mismatch. "
        "Set TOKENIZER_OVERRIDE explicitly."
    )

phase1_seq2seq_sanity()


tokenized 5000/32305 elapsed=15s
tokenized 10000/32305 elapsed=26s
tokenized 15000/32305 elapsed=39s
tokenized 20000/32305 elapsed=53s
tokenized 25000/32305 elapsed=65s
tokenized 30000/32305 elapsed=76s
done 32305 in 81s | est. median BPE=4416 p95=20119
   chunk-selection stats: {'long_doc': 15940, 'cue_found': 12469, 'cue_missing': 3471, 'short_doc': 16365}
done 994 in 3s | est. median BPE=6232 p95=21137
   chunk-selection stats: {'long_doc': 657, 'cue_found': 480, 'cue_missing': 177, 'short_doc': 337}
done 1517 in 5s | est. median BPE=6505 p95=22861
   chunk-selection stats: {'long_doc': 1053, 'cue_found': 772, 'short_doc': 464, 'cue_missing': 281}
batches: 16153 166 253
✅ Model3 encoder→decoder-summary→hierarchical-head route PASS
Phase-1 Model3 seq2seq sanity: loss=3.063 | ppl=21.4 | chance ln(V)=10.11


In [8]:
# CELL 8 — optimizer, exact metric, threshold search
counts=Counter(r["label"] for r in train_records)
_raw_w=np.array([len(train_records)/(2*counts[i]) for i in [0,1]],dtype=np.float64)
_raw_w=np.power(_raw_w,CLASS_WEIGHT_POWER); _raw_w=_raw_w/_raw_w.mean()
class_w=torch.tensor(_raw_w,dtype=torch.float32,device=DEVICE)
print("class weights:",class_w.tolist(),"| power:",CLASS_WEIGHT_POWER)

def make_groups(m):
    adapters=[
        p for n,p in m.named_parameters()
        if p.requires_grad and "adapter." in n
    ]
    head=[
        p for n,p in m.named_parameters()
        if p.requires_grad and "adapter." not in n
    ]
    assert adapters and head
    return [
        {"params":adapters,"lr":ADAPTER_LR,"weight_decay":0.0,"name":"adapters"},
        {"params":head,"lr":HEAD_LR,"weight_decay":WEIGHT_DECAY,"name":"head"},
    ]

groups=make_groups(model)
try:
    optimizer=torch.optim.AdamW(groups,fused=(DEVICE.type=="cuda"))
except Exception:
    optimizer=torch.optim.AdamW(groups)

updates_per_epoch=math.ceil(len(train_loader)/GRAD_ACCUM_STEPS)
total_updates=NUM_EPOCHS*updates_per_epoch
warmup=max(1,int(total_updates*WARMUP_RATIO))

def lr_mult(step):
    if step<warmup:
        return max(1e-8,step/warmup)
    p=(step-warmup)/max(1,total_updates-warmup)
    return .5*(1+math.cos(math.pi*min(max(p,0),1)))

scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_mult)
amp_enabled=USE_AMP and DEVICE.type=="cuda"
scaler=torch.amp.GradScaler(
    "cuda",enabled=amp_enabled,init_scale=AMP_INIT_SCALE,growth_interval=2000
)

def macro_f1(y,p):
    return f1_score(y,p,average="macro",labels=[0,1],zero_division=0)

def search_threshold(y,prob,smooth=THRESH_SMOOTH):
    fs=np.array([macro_f1(y,(prob>=t).astype(np.int64)) for t in THRESHOLDS])
    k=max(1,int(smooth))
    if k>1:
        if k%2==0:k+=1
        pad=k//2
        sm=np.convolve(np.pad(fs,(pad,pad),mode="edge"),np.ones(k)/k,mode="valid")
    else:
        sm=fs
    i=int(np.argmax(sm))
    t=float(THRESHOLDS[i])
    return float(fs[i]),t,(prob>=t).astype(np.int64),float(fs.max())

@torch.no_grad()
def predict_logits(m,loader):
    was=m.training
    m.eval()
    lgs=[]; ys=[]; ids=[]; ats=[]
    for b in loader:
        x=b["input_ids"].to(DEVICE,non_blocking=True)
        v=b["valid"].to(DEVICE,non_blocking=True)
        with torch.amp.autocast("cuda",dtype=torch.float16,enabled=amp_enabled):
            lg,a=m(x,v)
        lgs.append(lg.float().cpu().numpy())
        ats.append(a.float().cpu().numpy())
        ids.extend(b["ids"])
        if "labels" in b: ys.append(b["labels"].numpy())
    m.train(was)
    return np.concatenate(lgs),(np.concatenate(ys) if ys else None),ids,np.concatenate(ats)

def _p1(lg):
    lg=lg.astype(np.float64); lg=lg-lg.max(-1,keepdims=True); e=np.exp(lg)
    return (e[:,1]/e.sum(-1)).astype(np.float32)

def _logit(p):
    p=np.clip(np.asarray(p,dtype=np.float64),1e-6,1-1e-6)
    return np.log(p/(1-p))

def predict_probs(m,loader):
    lg,y,ids,att=predict_logits(m,loader)
    return _p1(lg),y,ids,att

print("updates/epoch:",updates_per_epoch,"| total updates:",total_updates)
print("trainable adapter params:",
      sum(p.numel() for n,p in model.named_parameters() if p.requires_grad and "adapter." in n))


class weights: [0.9136940240859985, 1.0863059759140015] | power: 0.5
updates/epoch: 2020 | total updates: 6060
trainable adapter params: 5973120


In [10]:
# ============================================================
# RESUME EXP5 FROM SAVED best_adapters_cjpe.pt
# Run this AFTER Cells 1–8.
# Do NOT run the training Cell 9.
# ============================================================

import os
import torch


# ------------------------------------------------------------
# Helpers originally defined at the top of training Cell 9
# ------------------------------------------------------------
def trainable_state_dict(m):
    names = {
        n for n, p in m.named_parameters()
        if p.requires_grad
    }

    sd = m.state_dict()

    return {
        k: v.detach().cpu()
        for k, v in sd.items()
        if k in names
    }


def load_trainable_state(m, state):
    expected = {
        n for n, p in m.named_parameters()
        if p.requires_grad
    }

    assert set(state) == expected, (
        f"trainable state mismatch:\n"
        f"missing={sorted(expected - set(state))[:10]}\n"
        f"extra={sorted(set(state) - expected)[:10]}"
    )

    own = m.state_dict()

    with torch.no_grad():
        for k, v in state.items():
            own[k].copy_(
                v.to(
                    device=own[k].device,
                    dtype=own[k].dtype
                )
            )


# ------------------------------------------------------------
# Locate saved checkpoint
# ------------------------------------------------------------
resume_candidates = []

for base in ["/kaggle/input", "/kaggle/working"]:
    if os.path.isdir(base):
        for root, _, files in os.walk(base):
            for fn in files:
                if fn == "best_adapters_cjpe.pt":
                    resume_candidates.append(
                        os.path.join(root, fn)
                    )

assert resume_candidates, (
    "Could not find best_adapters_cjpe.pt. "
    "Attach the saved checkpoint dataset."
)

# Prefer your attached checkpoint
preferred = [
    p for p in resume_candidates
    if "phase2-model3-ckpts" in p
]

BEST = preferred[0] if preferred else resume_candidates[0]

print("✅ Resuming from:")
print(BEST)


# ------------------------------------------------------------
# Load checkpoint
# ------------------------------------------------------------
best_obj = torch.load(
    BEST,
    map_location="cpu",
    weights_only=False
)

print("best DEV Macro-F1 :", best_obj["best_macro_f1"])
print("best threshold    :", best_obj["best_threshold"])
print("best epoch frac   :", best_obj["best_epoch_frac"])


# ------------------------------------------------------------
# Restore trained adapters + hierarchical CJPE head
# ------------------------------------------------------------
load_trainable_state(
    model,
    best_obj["trainable_state"]
)

best_f1  = float(best_obj["best_macro_f1"])
best_thr = float(best_obj["best_threshold"])
best_frac = float(best_obj["best_epoch_frac"])

print("\n✅ Best trained Model3 + Adapters state restored.")
print(f"DEV Macro-F1 = {best_f1:.6f}")
print(f"Threshold    = {best_thr:.3f}")
print(f"Epoch frac   = {best_frac:.3f}")

✅ Resuming from:
/kaggle/input/datasets/aaryant23/phase2-model3-ckpts/best_adapters_cjpe.pt
best DEV Macro-F1 : 0.5539462821428391
best threshold    : 0.405
best epoch frac   : 3.0

✅ Best trained Model3 + Adapters state restored.
DEV Macro-F1 = 0.553946
Threshold    = 0.405
Epoch frac   = 3.000


In [9]:
# CELL 9 — adapter training + CLEAN DEV selection
BEST=ROOT/"best_adapters_cjpe.pt"
BEST_TMP=BEST.with_suffix(".pt.tmp")
if BEST_TMP.exists(): BEST_TMP.unlink()

def trainable_state_dict(m):
    names={n for n,p in m.named_parameters() if p.requires_grad}
    sd=m.state_dict()
    return {k:v.detach().cpu() for k,v in sd.items() if k in names}

def load_trainable_state(m,state):
    expected={n for n,p in m.named_parameters() if p.requires_grad}
    assert set(state)==expected,(
        f"trainable state mismatch: missing={sorted(expected-set(state))[:5]} "
        f"extra={sorted(set(state)-expected)[:5]}"
    )
    own=m.state_dict()
    with torch.no_grad():
        for k,v in state.items():
            own[k].copy_(v.to(device=own[k].device,dtype=own[k].dtype))

history=[]; best_f1=-1.0; best_thr=.5; best_frac=None
bad=0; global_update=0; overflows=0; nonfinite_skipped=0
optimizer.zero_grad(set_to_none=True)

eval_steps=[]
for j in range(1,EVAL_PER_EPOCH+1):
    s=int(round(len(train_loader)*j/EVAL_PER_EPOCH))
    if s<len(train_loader):
        s=max(GRAD_ACCUM_STEPS,(s//GRAD_ACCUM_STEPS)*GRAD_ACCUM_STEPS)
    else:
        s=len(train_loader)
    eval_steps.append(s)
eval_steps=sorted(set(eval_steps))
print("DEV eval batches:",eval_steps)

stop=False
for epoch in range(1,NUM_EPOCHS+1):
    model.train(); run=0.; nr=0; ep0=time.time(); good=0
    for step,b in enumerate(train_loader,1):
        x=b["input_ids"].to(DEVICE,non_blocking=True)
        v=b["valid"].to(DEVICE,non_blocking=True)
        y=b["labels"].to(DEVICE,non_blocking=True)

        with torch.amp.autocast("cuda",dtype=torch.float16,enabled=amp_enabled):
            lg,_=model(x,v)
            raw=F.cross_entropy(
                lg.float(),y,weight=class_w,label_smoothing=LABEL_SMOOTHING
            )
            loss=raw/GRAD_ACCUM_STEPS

        raw_val=float(raw.detach())
        if math.isfinite(raw_val):
            scaler.scale(loss).backward()
            good+=1; run+=raw_val; nr+=1
        else:
            nonfinite_skipped+=1
            print(f"⚠️ non-finite loss epoch={epoch} batch={step}; skipped")

        if step%GRAD_ACCUM_STEPS==0 or step==len(train_loader):
            if good==0:
                optimizer.zero_grad(set_to_none=True)
            else:
                scaler.unscale_(optimizer)
                gn=torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad],GRAD_CLIP
                )
                finite=bool(torch.isfinite(torch.as_tensor(gn)).item())
                old=float(scaler.get_scale())

                if not finite:
                    optimizer.zero_grad(set_to_none=True)
                    if amp_enabled:
                        scaler.update(new_scale=max(old*.5,1.0))
                    overflows+=1
                else:
                    scaler.step(optimizer)
                    scaler.update()
                    new=float(scaler.get_scale())
                    optimizer.zero_grad(set_to_none=True)
                    scheduler.step(); global_update+=1
                    if new<old: overflows+=1
            good=0

        if step%500==0 or step==len(train_loader):
            elapsed=time.time()-ep0
            eta=(elapsed/max(step,1))*(len(train_loader)-step)
            print(
                f"epoch={epoch} batch={step}/{len(train_loader)} "
                f"loss={run/max(nr,1):.4f} elapsed={elapsed/60:.1f}m "
                f"ETA_epoch={eta/60:.1f}m "
                f"lr_adapter={optimizer.param_groups[0]['lr']:.2e}",
                flush=True
            )

        if step in eval_steps:
            prob,ydev,idsdev,att=predict_probs(model,dev_loader)
            f,t,pred,grid_best=search_threshold(ydev,prob)
            prob_diag={
                "prob_mean":float(prob.mean()),"prob_std":float(prob.std()),
                "prob_min":float(prob.min()),"prob_max":float(prob.max()),
                "positive_rate_thr":float((prob>=t).mean()),
                "positive_rate_05":float((prob>=.5).mean()),
                "decoder_gate":float(torch.sigmoid(model.decoder_gate_logit).detach().cpu()),
            }
            row={
                "epoch":epoch,
                "epoch_frac":(epoch-1)+step/len(train_loader),
                "batch":step,
                "macro_f1":f,
                "threshold":t,
                "grid_best_macro_f1":grid_best,
                "f1_at_0.5":float(macro_f1(ydev,(prob>=.5).astype(np.int64))),
                "train_loss":run/max(nr,1),
                "global_update":global_update,
                "amp_overflows":overflows,
                "nonfinite_skipped":nonfinite_skipped,
                "attention_mean":att.mean(0).tolist(),
                **prob_diag,
            }
            history.append(row)
            save_json(history,ROOT/"history.json",indent=2)
            print("DEV:",row)
            if prob_diag["prob_std"]<0.01:
                print(f"⚠️ DEV probabilities still narrow (std={prob_diag['prob_std']:.4f}).")
            if prob_diag["positive_rate_thr"] in (0.0,1.0):
                print("⚠️ Thresholded DEV prediction is currently one-class.")

            if f>best_f1+1e-12:
                best_f1=f; best_thr=t; best_frac=row["epoch_frac"]; bad=0
                torch.save({
                    "trainable_state":trainable_state_dict(model),
                    "best_macro_f1":float(best_f1),
                    "best_threshold":float(best_thr),
                    "best_epoch_frac":float(best_frac),
                    "adapter_type":"Houlsby bottleneck encoder+decoder",
                    "adapter_bottleneck":ADAPTER_BOTTLENECK,
                    "adapter_dropout":ADAPTER_DROPOUT,
                    "adapter_lr":ADAPTER_LR,
                    "head_lr":HEAD_LR,
                    "seed":SEED,
                    "method":"Adapters",
                },BEST_TMP)
                os.replace(BEST_TMP,BEST)
                print(f"✅ NEW BEST Macro-F1={best_f1:.6f} threshold={best_thr:.3f}")
            else:
                bad+=1

            if bad>=PATIENCE_EVALS:
                print("Early stopping.")
                stop=True
                break
            model.train()
    if stop: break

assert BEST.is_file()
print("DONE best F1/thr/frac:",best_f1,best_thr,best_frac)


DEV eval batches: [4032, 8072, 12112, 16153]
epoch=1 batch=500/16153 loss=0.6955 elapsed=0.7m ETA_epoch=21.9m lr_adapter=5.12e-05
epoch=1 batch=1000/16153 loss=0.7004 elapsed=1.4m ETA_epoch=21.4m lr_adapter=1.03e-04
epoch=1 batch=1500/16153 loss=0.7017 elapsed=2.1m ETA_epoch=20.7m lr_adapter=1.55e-04
epoch=1 batch=2000/16153 loss=0.7022 elapsed=2.8m ETA_epoch=20.0m lr_adapter=2.07e-04
epoch=1 batch=2500/16153 loss=0.7030 elapsed=3.5m ETA_epoch=19.2m lr_adapter=2.58e-04
epoch=1 batch=3000/16153 loss=0.7044 elapsed=4.2m ETA_epoch=18.5m lr_adapter=3.00e-04
epoch=1 batch=3500/16153 loss=0.7034 elapsed=4.9m ETA_epoch=17.8m lr_adapter=3.00e-04
epoch=1 batch=4000/16153 loss=0.7006 elapsed=5.6m ETA_epoch=17.1m lr_adapter=3.00e-04
DEV: {'epoch': 1, 'epoch_frac': 0.2496130749705937, 'batch': 4032, 'macro_f1': 0.4421673923560716, 'threshold': 0.445, 'grid_best_macro_f1': 0.5501993802458163, 'f1_at_0.5': 0.3333333333333333, 'train_loss': 0.7005596318474364, 'global_update': 504, 'amp_overflows': 0

In [11]:
# CELL 10 — exact supplied Phase-4 evaluator on clean DEV
best_obj=torch.load(BEST,map_location="cpu",weights_only=False)
load_trainable_state(model,best_obj["trainable_state"])
best_thr=float(best_obj["best_threshold"])

dev_prob,dev_y,dev_ids,dev_att=predict_probs(model,dev_loader)
dev_pred=(dev_prob>=best_thr).astype(np.int64)

truth=[{"id":str(i),"gold":int(y)} for i,y in zip(dev_ids,dev_y)]
pred=[{"id":str(i),"pred":int(p)} for i,p in zip(dev_ids,dev_pred)]

EVALUATOR_CODE=r'''#!/usr/bin/env python3
import argparse,json,sys
from sklearn.metrics import f1_score,precision_score,recall_score
def read_truth(path):
    try:
        x=json.load(open(path,encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")
    if not isinstance(x,list) or not x: raise ValueError('Truth must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'gold' not in r: raise ValueError(f'Truth item {n} must contain id and gold.')
        if not isinstance(r['id'],str) or not r['id'].strip() or isinstance(r['gold'],bool) or r['gold'] not in (0,1): raise ValueError(f'Invalid truth item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate truth ID: {r["id"]}')
        out[r['id']]=r['gold']
    return out
def read_pred(path):
    x=json.load(open(path,encoding='utf-8'))
    if not isinstance(x,list) or not x: raise ValueError('Prediction must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'pred' not in r: raise ValueError(f'Prediction item {n} must contain id and pred.')
        if not isinstance(r['id'],str) or isinstance(r['pred'],bool) or r['pred'] not in (0,1): raise ValueError(f'Invalid prediction item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate prediction ID: {r["id"]}')
        out[r['id']]=r['pred']
    return out
def main():
    ap=argparse.ArgumentParser()
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True)
    a=ap.parse_args()
    t=read_truth(a.truth); p=read_pred(a.prediction)
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    yt=[t[i] for i in t]; yp=[p[i] for i in t]
    print(f'Examples: {len(t)}')
    print(f'Macro-F1: {f1_score(yt,yp,average="macro",labels=[0,1],zero_division=0):.6f}')
    print(f'Precision: {precision_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
    print(f'Recall: {recall_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e:
        print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)
'''

with tempfile.TemporaryDirectory() as td:
    tp=Path(td)/"truth.json"
    pp=Path(td)/"prediction.json"
    ep=Path(td)/"evaluate_phase4.py"
    save_json(truth,tp)
    save_json(pred,pp)
    ep.write_text(EVALUATOR_CODE,encoding="utf-8")
    res=subprocess.run(
        [sys.executable,str(ep),"--truth",str(tp),"--prediction",str(pp)],
        capture_output=True,text=True,
    )
    print(res.stdout)
    if res.stderr: print(res.stderr)
    assert res.returncode==0

print("✅ Exact Phase-4 evaluator PASS")
np.savez_compressed(
    ROOT/"dev_probs.npz",
    prob=dev_prob.astype(np.float32),
    y=dev_y.astype(np.int64),
    id=np.asarray(dev_ids,dtype=str),   # loads without allow_pickle
    threshold=np.asarray([best_thr],dtype=np.float32),
    chunk_attention=dev_att.astype(np.float32),
)


Examples: 994
Macro-F1: 0.553946
Precision: 0.557692
Recall: 0.525151

✅ Exact Phase-4 evaluator PASS


In [12]:
# CELL 11 — fresh Model3+Adapters and clean TRAIN+DEV refit
def fresh_model():
    try:
        o=torch.load(PHASE1_CKPT,map_location="cpu",weights_only=True)
    except Exception:
        o=torch.load(PHASE1_CKPT,map_location="cpu",weights_only=False)
    b=build_backbone()
    b.load_state_dict(o["model"],strict=True)
    n=inject_adapters(b)
    assert n==45
    return HierarchicalCJPE(b).to(DEVICE)

refit_model=None
REFIT_SEED=SEED+1

if RUN_REFIT:
    all_records=train_records+clean_dev
    all_ds=torch.utils.data.ConcatDataset([train_ds,dev_ds])

    seed_everything(REFIT_SEED)
    gg=torch.Generator(); gg.manual_seed(REFIT_SEED)
    all_loader=DataLoader(
        all_ds,batch_size=MICRO_BATCH_SIZE,shuffle=True,generator=gg,
        collate_fn=collate_cases,num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type=="cuda",persistent_workers=NUM_WORKERS>0,
    )

    refit_model=fresh_model()
    try:
        ropt=torch.optim.AdamW(make_groups(refit_model),fused=(DEVICE.type=="cuda"))
    except Exception:
        ropt=torch.optim.AdamW(make_groups(refit_model))

    target_batches=max(
        GRAD_ACCUM_STEPS,
        int(round(float(best_obj["best_epoch_frac"])*len(all_loader)/GRAD_ACCUM_STEPS))
        *GRAD_ACCUM_STEPS,
    )
    upe_all=math.ceil(len(all_loader)/GRAD_ACCUM_STEPS)
    horizon_updates=(
        NUM_EPOCHS*upe_all
        if REFIT_MATCH_ORIGINAL_SCHEDULE
        else math.ceil(target_batches/GRAD_ACCUM_STEPS)
    )
    rw=max(1,int(horizon_updates*WARMUP_RATIO))

    def rmult(step):
        if step<rw: return max(1e-8,step/rw)
        p=(step-rw)/max(1,horizon_updates-rw)
        return .5*(1+math.cos(math.pi*min(max(p,0),1)))

    rsched=torch.optim.lr_scheduler.LambdaLR(ropt,rmult)
    rscaler=torch.amp.GradScaler(
        "cuda",enabled=amp_enabled,init_scale=AMP_INIT_SCALE,growth_interval=2000
    )

    ac=Counter(r["label"] for r in all_records)
    cw=torch.tensor(
        [len(all_records)/(2*ac[i]) for i in [0,1]],
        dtype=torch.float32,device=DEVICE
    )
    cw/=cw.mean()

    print("refit target micro-batches:",target_batches,"| LR horizon updates:",horizon_updates)
    ropt.zero_grad(set_to_none=True)
    seen=0; rupd=0; over=0; nonfin=0; good=0; t0=time.time()

    while seen<target_batches:
        refit_model.train()
        for b in all_loader:
            if seen>=target_batches: break
            x=b["input_ids"].to(DEVICE,non_blocking=True)
            v=b["valid"].to(DEVICE,non_blocking=True)
            y=b["labels"].to(DEVICE,non_blocking=True)

            with torch.amp.autocast("cuda",dtype=torch.float16,enabled=amp_enabled):
                lg,_=refit_model(x,v)
                raw=F.cross_entropy(
                    lg.float(),y,weight=cw,label_smoothing=LABEL_SMOOTHING
                )
                loss=raw/GRAD_ACCUM_STEPS

            seen+=1
            if math.isfinite(float(raw.detach())):
                rscaler.scale(loss).backward(); good+=1
            else:
                nonfin+=1

            if seen%GRAD_ACCUM_STEPS==0 or seen>=target_batches:
                if good==0:
                    ropt.zero_grad(set_to_none=True)
                else:
                    rscaler.unscale_(ropt)
                    gn=torch.nn.utils.clip_grad_norm_(
                        [p for p in refit_model.parameters() if p.requires_grad],GRAD_CLIP
                    )
                    finite=bool(torch.isfinite(torch.as_tensor(gn)).item())
                    old=float(rscaler.get_scale())

                    if not finite:
                        ropt.zero_grad(set_to_none=True)
                        if amp_enabled:
                            rscaler.update(new_scale=max(old*.5,1.0))
                        over+=1
                    else:
                        rscaler.step(ropt); rscaler.update()
                        new=float(rscaler.get_scale())
                        ropt.zero_grad(set_to_none=True)
                        rsched.step(); rupd+=1
                        if new<old: over+=1
                good=0

            if seen%500==0 or seen>=target_batches:
                rate=(time.time()-t0)/max(seen,1)
                print(
                    f"refit {seen}/{target_batches} "
                    f"ETA={(target_batches-seen)*rate/60:.1f}m overflow={over}",
                    flush=True
                )

    FINAL=ROOT/"final_refit_adapters_cjpe.pt"
    TMP=FINAL.with_suffix(".pt.tmp")
    if TMP.exists(): TMP.unlink()
    torch.save({
        "trainable_state":trainable_state_dict(refit_model),
        "source_best_epoch_frac":float(best_obj["best_epoch_frac"]),
        "adapter_type":"Houlsby bottleneck encoder+decoder",
        "adapter_bottleneck":ADAPTER_BOTTLENECK,
        "adapter_dropout":ADAPTER_DROPOUT,
        "adapter_lr":ADAPTER_LR,
        "head_lr":HEAD_LR,
        "refit_seed":REFIT_SEED,
        "refit_target_batches":int(target_batches),
        "overflows":over,
        "nonfinite_skipped":nonfin,
        "method":"Adapters",
    },TMP)
    os.replace(TMP,FINAL)
    print("✅ refit complete")


refit target micro-batches: 49952 | LR horizon updates: 6246
refit 500/49952 ETA=68.3m overflow=0
refit 1000/49952 ETA=68.0m overflow=0
refit 1500/49952 ETA=67.0m overflow=0
refit 2000/49952 ETA=66.3m overflow=0
refit 2500/49952 ETA=65.6m overflow=0
refit 3000/49952 ETA=64.9m overflow=0
refit 3500/49952 ETA=64.2m overflow=0
refit 4000/49952 ETA=63.5m overflow=0
refit 4500/49952 ETA=62.8m overflow=0
refit 5000/49952 ETA=62.1m overflow=0
refit 5500/49952 ETA=61.4m overflow=0
refit 6000/49952 ETA=60.7m overflow=0
refit 6500/49952 ETA=60.0m overflow=0
refit 7000/49952 ETA=59.3m overflow=0
refit 7500/49952 ETA=58.6m overflow=0
refit 8000/49952 ETA=57.9m overflow=0
refit 8500/49952 ETA=57.2m overflow=0
refit 9000/49952 ETA=56.5m overflow=0
refit 9500/49952 ETA=55.8m overflow=0
refit 10000/49952 ETA=55.1m overflow=0
refit 10500/49952 ETA=54.4m overflow=0
refit 11000/49952 ETA=53.7m overflow=0
refit 11500/49952 ETA=53.0m overflow=0
refit 12000/49952 ETA=52.3m overflow=0
refit 12500/49952 ETA=5

In [13]:
# CELL 12 — TEST inference + label-free calibration
held_lg,_,held_ids,held_att=predict_logits(model,test_loader)
assert held_ids==[r["id"] for r in test_records] and len(held_ids)==1517
held_prob=_p1(held_lg)

refit_prob=None; refit_thr=None; test_refit_agreement=None
if refit_model is not None:
    refit_lg,_,rid,_=predict_logits(refit_model,test_loader)
    assert rid==held_ids
    refit_prob=_p1(refit_lg)
    held_rate=float((held_prob>=best_thr).mean())
    refit_thr=float(np.clip(np.quantile(refit_prob,1-held_rate),.05,.95))
    test_refit_agreement=float(((held_prob>=best_thr)==(refit_prob>=refit_thr)).mean())
    print(f"heldout positive rate={held_rate:.3f} | refit thr={refit_thr:.3f} | agreement={test_refit_agreement:.4f}")

np.savez_compressed(
    ROOT/"test_probs.npz",heldout=held_prob.astype(np.float32),
    refit=(refit_prob.astype(np.float32) if refit_prob is not None else np.array([],dtype=np.float32)),
    id=np.asarray(held_ids,dtype=str),
    heldout_threshold=np.asarray([best_thr],dtype=np.float32),
    refit_threshold=np.asarray([refit_thr if refit_thr is not None else np.nan],dtype=np.float32),
)


heldout positive rate=0.496 | refit thr=0.438 | agreement=0.9176


In [14]:
# CELL 13 — three high-value Codabench ZIPs
official_ids=[r["id"] for r in test_records]
assert len(official_ids)==len(set(official_ids))==1517

def write_zip(filename,labels):
    labels=np.asarray(labels,dtype=np.int64)
    rec=[{"id":str(cid),"pred":int(y)} for cid,y in zip(official_ids,labels)]
    assert len(rec)==1517 and len({r["id"] for r in rec})==1517
    assert all(type(r["pred"]) is int and r["pred"] in (0,1) for r in rec)
    zp=ROOT/filename
    with tempfile.TemporaryDirectory() as td:
        jp=Path(td)/"prediction.json"
        save_json(rec,jp)
        with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z:
            z.write(jp,arcname="prediction.json")
    with zipfile.ZipFile(zp,"r") as z:
        assert z.namelist()==["prediction.json"],z.namelist()
        inside=json.loads(z.read("prediction.json").decode("utf-8"))
        assert len(inside)==1517
        assert {r["id"] for r in inside}==set(official_ids)
    print(f"✅ {filename} | positive={labels.mean():.4f} | {Counter(labels.tolist())}")
    return labels

if refit_prob is not None:
    margin=.5*(_logit(held_prob)-_logit(best_thr))+.5*(_logit(refit_prob)-_logit(refit_thr))
    y_best=write_zip("submission_best.zip",(margin>=0).astype(np.int64))
else:
    y_best=write_zip("submission_best.zip",(held_prob>=best_thr).astype(np.int64))

y_held=write_zip("submission_heldout.zip",(held_prob>=best_thr).astype(np.int64))

if refit_prob is not None:
    margin_h70_r30=.70*(_logit(held_prob)-_logit(best_thr)) + .30*(_logit(refit_prob)-_logit(refit_thr))
    y_hedge=write_zip("submission_margin_h70_r30.zip",(margin_h70_r30>=0).astype(np.int64))
else:
    y_hedge=None

disagree_best_held=float(np.mean(y_best!=y_held))
disagree_best_hedge=(None if y_hedge is None else float(np.mean(y_best!=y_hedge)))
disagree_held_hedge=(None if y_hedge is None else float(np.mean(y_held!=y_hedge)))
print("best vs heldout disagreement:",disagree_best_held)
if y_hedge is not None:
    print("best vs 70/30 hedge disagreement:",disagree_best_hedge)
    print("heldout vs 70/30 hedge disagreement:",disagree_held_hedge)


✅ submission_best.zip | positive=0.4964 | Counter({0: 764, 1: 753})
✅ submission_heldout.zip | positive=0.4964 | Counter({0: 764, 1: 753})
✅ submission_margin_h70_r30.zip | positive=0.4964 | Counter({0: 764, 1: 753})
best vs heldout disagreement: 0.03691496374423204
best vs 70/30 hedge disagreement: 0.014502307185234015
heldout vs 70/30 hedge disagreement: 0.02241265655899802


In [15]:
# CELL 14 — manifest
manifest={
    "phase":4,
    "experiment":5,
    "notebook_version":NOTEBOOK_VERSION,
    "method":"Post-residual Houlsby-style bottleneck adapters",
    "phase1_model":"Model3 encoder-decoder Transformer",
    "checkpoint":str(PHASE1_CKPT),
    "checkpoint_sha256":sha256_file(PHASE1_CKPT),
    "tokenizer_sha256":sha256_file(TOKENIZER_FILE),
    "dataset_sha256":DATA_SHA256,
    "raw_dataset_rows":RAW_ROW_COUNTS,

    "architecture":{
        "vocab_size":VOCAB_SIZE,
        "d_model":D_MODEL,
        "encoder_layers":N_ENCODER_LAYERS,
        "decoder_layers":N_DECODER_LAYERS,
        "n_heads":N_HEADS,
        "ffn_dim":FFN_DIM,
        "max_encoder_positions":MAX_ENCODER_POSITIONS,
        "max_decoder_positions":MAX_DECODER_POSITIONS,
    },

    "train_unique":len(train_records),
    "clean_dev":len(clean_dev),
    "test":len(test_records),
    "excluded_dev_test_overlap":overlap_count,

    "chunks":CHUNK_NAMES,
    "chunk_len":CHUNK_LEN,
    "chunk_representation":"encoder last + encoder mean + masked max + learnably gated decoder cross-attention summary",

    "adapter_type":"post-residual Houlsby bottleneck + adapter LayerNorm + zero-init up projection",
    "adapter_scope":"all encoder and decoder sublayers",
    "adapter_count":N_ADAPTERS,
    "adapter_bottleneck":ADAPTER_BOTTLENECK,
    "adapter_dropout":ADAPTER_DROPOUT,
    "adapter_lr":ADAPTER_LR,
    "head_lr":HEAD_LR,
    "class_weight_power":CLASS_WEIGHT_POWER,
    "label_smoothing":LABEL_SMOOTHING,

    "best_dev_macro_f1":float(best_obj["best_macro_f1"]),
    "best_threshold":float(best_obj["best_threshold"]),
    "best_epoch_frac":float(best_obj["best_epoch_frac"]),

    "lowercase":bool(LOWERCASE),
    "collapse_ws":bool(COLLAPSE_WS),

    "refit_run":bool(refit_model is not None),
    "refit_threshold":None if refit_thr is None else float(refit_thr),
    "test_refit_agreement":test_refit_agreement,
    "refit_target_batches":int(target_batches) if refit_model is not None else None,

    "test_positive_rate_best":float(np.mean(y_best)),
    "submission_strategies":{
        "submission_best.zip":"50/50 calibrated heldout+refit margin ensemble",
        "submission_heldout.zip":"clean-DEV-selected heldout checkpoint",
        "submission_margin_h70_r30.zip":"70/30 calibrated heldout/refit margin hedge",
    },
    "submission_disagreement":{
        "best_vs_heldout":disagree_best_held,
        "best_vs_h70_r30":disagree_best_hedge,
        "heldout_vs_h70_r30":disagree_held_hedge,
    },

    "torch_version":torch.__version__,
    "seed":SEED,
    "test_labels_used":False,
    "overlap_dev_labels_used":False,
}
save_json(manifest,ROOT/"manifest.json",indent=2)
print(json.dumps(manifest,indent=2))

print("\nESSENTIAL OUTPUTS:")
for fn in [
    "best_adapters_cjpe.pt",
    "final_refit_adapters_cjpe.pt",
    "dev_probs.npz",
    "test_probs.npz",
    "history.json",
    "manifest.json",
    "submission_best.zip",
    "submission_heldout.zip",
    "submission_margin_h70_r30.zip",
]:
    print(("✅" if (ROOT/fn).exists() else "—"),fn)


{
  "phase": 4,
  "experiment": 5,
  "notebook_version": "v4_BETTER",
  "method": "Post-residual Houlsby-style bottleneck adapters",
  "phase1_model": "Model3 encoder-decoder Transformer",
  "checkpoint": "/kaggle/input/datasets/aaryant23/phase2-model3-ckpts/encdec_full_best (2).pt",
  "checkpoint_sha256": "28754f7ae29ab3e2c5789b979f7f3947c64d20052bf083a8899432f05a40d694",
  "tokenizer_sha256": "9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d",
  "dataset_sha256": {
    "train": "a115cc722e9de8b68ac5d41179c12d86c2d537c0de8ff16412c44c6c8d1cce72",
    "dev": "aa2d80ae77f5d532fbc3f30e4dea2776700b8e92e71c603737d8f41a9d30b514",
    "test": "f3498fb832e90fe7264fcb6770918eb5261409e9a952d47a2979b6ed5b153b59"
  },
  "raw_dataset_rows": {
    "train": 37387,
    "dev": 1988,
    "test": 1517
  },
  "architecture": {
    "vocab_size": 24576,
    "d_model": 512,
    "encoder_layers": 9,
    "decoder_layers": 9,
    "n_heads": 8,
    "ffn_dim": 2048,
    "max_encoder_positions": 20

## Keep / submit

Primary candidate: **`submission_best.zip`**.

Also keep `submission_heldout.zip` and `submission_margin_h70_r30.zip`.

During training, watch the DEV diagnostics. A healthy run should show a non-trivial `prob_std`, and neither `positive_rate_thr` nor `positive_rate_05` should stay permanently at 0 or 1.
